# Access lineage diagnostic

Diagnóstico **read-only**, determinístico e sem `gabarito.csv`. Este notebook não executa nem modifica generator, fontes, Bronze, Silver, Access Context, HTS, baseline, fallback ou observabilidade.

Contrato esperado de `iga_access_assignments`: `grant_id`, `identidade_id`, `entitlement_id`, `request_id`, `data_concessao`, `data_revogacao`, `status_concessao`, `tipo_atribuicao`, `ultimo_uso`. Grain: uma linha por grant. Uma chave técnica só é aceitável quando a origem não oferece `grant_id` utilizável.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from pyspark.sql import functions as F
from sod_platform.env import load_project_env

from sod_platform.bronze.ingestion.config import load_sources
from sod_platform.bronze.ingestion.discovery import discover_files
from sod_platform.bronze.ingestion.spark import create_spark_session

load_project_env(ROOT)
spark = create_spark_session(ROOT, "sod-access-lineage-diagnostic")
sources = {
    source.source_name: source for source in load_sources(ROOT / "configs/sources.yml")
}
assignment_source = sources["access_assignments"]
assignment_paths = discover_files(assignment_source.path, ROOT)
assert assignment_paths, f"No source files found for {assignment_source.path}"
assert all("gabarito" not in path.lower() for path in assignment_paths)
print(
    {
        "source_path_pattern": assignment_source.path,
        "source_files": len(assignment_paths),
        "bronze_table": assignment_source.target_table,
    }
)

In [ ]:
def profile_grants(frame, label, sample_columns):
    """Display schema, coverage and duplicate evidence without collecting full data."""
    print(f"\n## {label}")
    frame.printSchema()
    if "grant_id" not in frame.columns:
        print(
            f"CONTRACT VIOLATION: {label} has no grant_id; coverage, uniqueness and grant-ID trace are not computable."
        )
        frame.select(
            *[column for column in sample_columns if column in frame.columns]
        ).limit(20).show(truncate=False)
        return {
            "rows": frame.count(),
            "grant_id_present": False,
            "distinct_grant_id": None,
            "null_grant_id": None,
            "empty_grant_id": None,
            "duplicate_rows": None,
        }
    total = frame.count()
    summary = (
        frame.agg(
            F.countDistinct("grant_id").alias("distinct_grant_id"),
            F.sum(F.col("grant_id").isNull().cast("long")).alias("null_grant_id"),
            F.sum((F.trim(F.col("grant_id")) == "").cast("long")).alias(
                "empty_grant_id"
            ),
        )
        .first()
        .asDict()
    )
    duplicates = (
        frame.where(F.col("grant_id").isNotNull())
        .groupBy("grant_id")
        .count()
        .where("count > 1")
    )
    duplicate_rows = duplicates.agg(
        F.coalesce(F.sum("count"), F.lit(0)).alias("duplicate_rows")
    ).first()["duplicate_rows"]
    result = {"rows": total, **summary, "duplicate_rows": duplicate_rows}
    print(result)
    frame.select(
        *[column for column in sample_columns if column in frame.columns]
    ).orderBy("grant_id").limit(20).show(truncate=False)
    duplicates.orderBy(F.desc("count"), "grant_id").limit(20).show(truncate=False)
    return result


def grant_id_contract(frame, label):
    if "grant_id" not in frame.columns:
        print(f"CONTRACT VIOLATION: {label} has no grant_id.")
        return False
    bad = frame.where(
        F.col("grant_id").isNull() | (F.trim(F.col("grant_id")) == "")
    ).count()
    duplicated = (
        frame.groupBy("grant_id")
        .count()
        .where("grant_id IS NOT NULL AND count > 1")
        .count()
    )
    valid = bad == 0 and duplicated == 0
    if not valid:
        print(
            f"CONTRACT VIOLATION: {label} has null/empty={bad}, duplicate IDs={duplicated}; grain is not provably one row per grant."
        )
    return valid

## Parte A — lineage de `grant_id`

As próximas células mostram o schema real, cobertura, duplicidade, amostras e o trace Source → Bronze → Silver → Access Context.

In [ ]:
source_assignments = (
    spark.read.format(assignment_source.format)
    .options(**assignment_source.options)
    .load(assignment_paths)
)
bronze_assignments = spark.table("sod.bronze.access_assignments")
silver_assignments = spark.table("sod.silver.iga_access_assignments")
access_context = spark.table("sod.silver.access_context")

sample = [
    "grant_id",
    "identidade_id",
    "entitlement_id",
    "request_id",
    "data_concessao",
    "status_concessao",
    "_source_name",
    "_source_file",
    "_ingestion_id",
    "grant_id_generated",
    "grant_id_generation_method",
]
source_stats = profile_grants(
    source_assignments, "SOURCE: iga_access_assignments", sample
)
bronze_stats = profile_grants(bronze_assignments, "BRONZE: access_assignments", sample)
silver_stats = profile_grants(
    silver_assignments, "SILVER: iga_access_assignments", sample
)
context_stats = profile_grants(access_context, "ACCESS CONTEXT", sample)

if "grant_id_generated" in access_context.columns:
    access_context.groupBy(
        "grant_id_generated", "grant_id_generation_method"
    ).count().orderBy("grant_id_generated", "grant_id_generation_method").show(
        truncate=False
    )
else:
    print(
        "LIMITATION: Access Context snapshot has no generation flag; no generated=true conclusion is made."
    )

In [ ]:
# A deterministic, bounded source sample. Contract violations remain explicit.
source_grant_contract = grant_id_contract(source_assignments, "SOURCE")
bronze_grant_contract = grant_id_contract(bronze_assignments, "BRONZE")
silver_grant_contract = grant_id_contract(silver_assignments, "SILVER")
context_grant_contract = grant_id_contract(access_context, "ACCESS CONTEXT")
if not source_grant_contract:
    print(
        "TRACE LIMITATION: SOURCE has no usable grant_id, so a Source → Bronze grant-ID trace cannot be constructed without inventing a matching key. This is evidence for hypothesis A, not a correction."
    )
else:
    trace_ids = (
        source_assignments.select("grant_id").orderBy("grant_id").limit(15).cache()
    )
    try:

        def ids(frame, alias):
            extras = [
                column
                for column in ("grant_id_generated", "grant_id_generation_method")
                if column in frame.columns
            ]
            return frame.join(trace_ids, "grant_id", "inner").select(
                F.col("grant_id").alias(alias),
                *[F.col(column).alias(f"{alias}_{column}") for column in extras],
            )

        source_trace = ids(source_assignments, "source_grant_id")
        bronze_trace = ids(bronze_assignments, "bronze_grant_id")
        silver_trace = ids(silver_assignments, "silver_grant_id")
        context_trace = ids(access_context, "access_context_grant_id")
        trace = (
            source_trace.join(
                bronze_trace,
                source_trace.source_grant_id == bronze_trace.bronze_grant_id,
                "left",
            )
            .join(
                silver_trace,
                source_trace.source_grant_id == silver_trace.silver_grant_id,
                "left",
            )
            .join(
                context_trace,
                source_trace.source_grant_id == context_trace.access_context_grant_id,
                "left",
            )
        )
        trace = trace.select(
            "source_grant_id",
            "bronze_grant_id",
            "silver_grant_id",
            "access_context_grant_id",
            (F.col("source_grant_id") == F.col("bronze_grant_id")).alias(
                "preserved_source_to_bronze"
            ),
            (F.col("bronze_grant_id") == F.col("silver_grant_id")).alias(
                "preserved_bronze_to_silver"
            ),
            (F.col("silver_grant_id") == F.col("access_context_grant_id")).alias(
                "preserved_silver_to_context"
            ),
            *[
                column
                for column in context_trace.columns
                if "generated" in column or "generation_method" in column
            ],
        )
        assert trace.count() == 15, "Diagnostic join multiplied or lost sampled grants"
        trace.show(truncate=False)
    finally:
        trace_ids.unpersist()

print(
    {
        "GRANT ID DIAGNOSTIC": {
            "source": source_stats,
            "bronze": bronze_stats,
            "silver": silver_stats,
            "access_context": context_stats,
        }
    }
)

## Parte B — birthright e Hard Trusted Set

In [ ]:
entitlements = spark.table("sod.silver.iga_entitlements")
print("ENTITLEMENTS")
entitlements.printSchema()
entitlements.groupBy("birthright").count().orderBy("birthright").show(truncate=False)
entitlements.where("birthright = true").limit(20).show(truncate=False)
assert entitlements.groupBy("entitlement_id").count().where("count > 1").count() == 0, (
    "Entitlements are not unique; diagnostic join would multiply grants"
)

before = silver_assignments.count()
grant_birthright = silver_assignments.select(
    "grant_id", "identidade_id", "entitlement_id"
).join(entitlements.select("entitlement_id", "birthright"), "entitlement_id", "left")
after = grant_birthright.count()
distinct_after = grant_birthright.select("grant_id").distinct().count()
assert before == after == distinct_after, (
    "Birthright diagnostic join changed grant grain"
)
grant_birthright.groupBy("birthright").agg(
    F.count("*").alias("grants"), F.countDistinct("identidade_id").alias("identities")
).orderBy("birthright").show(truncate=False)

print(
    "birthright_eligible origin: Access Context creates it as F.lit(None).cast(boolean) in silver/access_context.py; it is not a source field, lookup or configured eligibility rule."
)
assert "birthright_eligible" in access_context.columns, (
    "Access Context lacks birthright_eligible required by HTS"
)
access_context.groupBy("birthright", "birthright_eligible").count().orderBy(
    "birthright", "birthright_eligible"
).show(truncate=False)

In [ ]:
hts_table = "sod.access_intelligence.hard_trusted_set"
baseline_table = "sod.access_intelligence.observed_baseline"
fallback_table = "sod.access_intelligence.hierarchical_fallback"
for table in (hts_table, baseline_table, fallback_table):
    assert spark.catalog.tableExists(table), (
        f"{table} is absent. Run the existing Access Intelligence CLI before this read-only diagnostic."
    )
hts = spark.table(hts_table)
hts_sample = hts.where("birthright = true").orderBy("grant_id").limit(20)
hts_sample.select(
    *[
        column
        for column in [
            "grant_id",
            "identidade_id",
            "entitlement_id",
            "comunidade",
            "cargo",
            "tipo_identidade",
            "birthright",
            "birthright_eligible",
            "cross_community",
            "approval_relevance",
            "certification_decisao",
            "hard_trusted_flag",
            "hard_trusted_reason",
        ]
        if column in hts.columns
    ]
).show(truncate=False)
hts.groupBy("hard_trusted_flag", "hard_trusted_reason").count().orderBy(
    "hard_trusted_flag", "hard_trusted_reason"
).show(truncate=False)

birthright_trace = (
    hts.where("birthright = true")
    .orderBy("grant_id")
    .select(
        *[
            column
            for column in [
                "grant_id",
                "entitlement_id",
                "birthright",
                "birthright_eligible",
                "hard_trusted_flag",
                "hard_trusted_reason",
            ]
            if column in hts.columns
        ]
    )
    .limit(20)
)
birthright_trace.show(truncate=False)
baseline = spark.table(baseline_table)
fallback = spark.table(fallback_table)
cascade = {
    "hts_evaluated": hts.count(),
    "hts_trusted": hts.where("hard_trusted_flag").count(),
    "baseline_records": baseline.count(),
    "fallback_evaluated": fallback.count(),
    "fallback_selected": fallback.where(
        "selection_status = 'BASELINE_SELECTED'"
    ).count(),
    "insufficient_evidence": fallback.where(
        "selection_status = 'INSUFFICIENT_EVIDENCE'"
    ).count(),
}
print(
    {
        "HTS DIAGNOSTIC": cascade,
        "birthright_eligible_origin": "attribute inexistente na fonte; Access Context o materializa como UNKNOWN/null",
    }
)
fallback.groupBy(
    "selection_status", "selected_baseline_level", "fallback_depth"
).count().orderBy("selection_status", "selected_baseline_level", "fallback_depth").show(
    truncate=False
)

# DIAGNOSTIC CONCLUSION

Execute as células acima em ordem. Elas imprimem as evidências necessárias para responder sem inferência: cobertura e preservação de `grant_id`, IDs efetivamente técnicos, origem de `birthright_eligible`, motivo do HTS e o efeito cascata até fallback.

O notebook deliberadamente não escolhe hipótese nem altera componentes: uma assertion interrompe a execução quando o contrato necessário não é atendido.